# **Unconditional *de novo* symmetric protein design with RFDiffusion3**

## Introduction

Many native proteins and enzymes assemble into higher-order complexes governed by well-defined point-group symmetries, which are often essential for their structural and catalytic functions. RFdiffusion3 (RFD3) extends its all-atom diffusion framework beyond single-chain monomers to unconditionally generate de novo symmetric homo-oligomers across both cyclic (Cn) and dihedral (Dn) point-group symmetries.

<figure>
<center>
<img src='https://raw.githubusercontent.com/pb3lab/ibm3410/main/img/rfd3_img02.png'/>
<figcaption><b>Fig. 1</b>. Unconditional symmetry design with RFD3, which can be further used to generate symmetric enzymes.
<br><i>Image Credits: Jasper Butcher</i></figcaption></center>
</figure>

RFD3 enforces symmetry directly within its reverse diffusion loop without requiring a separate model:
- **Asymmetric Unit (ASU) Symmetrization**: At each denoising step, the diffusion module predicts the denoised coordinates, extracts the asymmetric unit ($\hat{x}_0^{\text{asu}}$), and applies the point-group symmetry operations ($R = \{R_1, \dots, R_N\}$) to replicate and symmetrize the full complex.
- **Early Symmetrization Stop**: Symmetrization can be stopped before the end of the trajectory—by default after 90% of the denoising steps ($\sigma_{\text{sym}} = 0.9$)—allowing the final steps to refine local atomic geometry.

In this tutorial, we will explore how to perform unconditional _de novo_ protein design of symmetric homo-oligomers using **RFD3**, improve their sequences with **MPNN**, and _in silico_ validate that the sequences fold into the RFD3-designed symmetric oligomers in single sequence mode via one of many protein structure prediction architectures (including **AlphaFold3**, **OpenFold3**, **Boltz-2** and **RoseTTAFold3**) as an oracle.

In [ ]:
%%time
import os, sys, re, time, glob, json, shlex, shutil, pathlib, tempfile, warnings, subprocess
import urllib.request
from concurrent.futures import ThreadPoolExecutor, as_completed
warnings.filterwarnings('ignore')

try:
  from rich.console import Console
  from rich.progress import Progress, TextColumn, BarColumn, DownloadColumn, TransferSpeedColumn, TimeRemainingColumn
except ModuleNotFoundError:
  !pip -q install rich
  from rich.console import Console
  from rich.progress import Progress, TextColumn, BarColumn, DownloadColumn, TransferSpeedColumn, TimeRemainingColumn

#@title Setup environment
#@markdown Run this only **once**. Typically takes ~3 min.
#@markdown - **Foundry**: RFD3 + built-in MPNN (SolubleMPNN / LigandMPNN / ProteinMPNN) + optional RF3
#@markdown - **Folding model**: Boltz2, RosettaFold3, AlphaFold3, or OpenFold3

folding_model = "AlphaFold3" #@param ["RosettaFold3", "Boltz2", "AlphaFold3", "OpenFold3"] {type:"string"}

#@markdown Save Foundry checkpoints (RFD3, MPNN, RF3) to Google Drive to skip re-downloading across sessions
persist_foundry_checkpoints_to_drive = False #@param {type:"boolean"}
#@markdown Save AF3/OpenFold3 compiled XLA cache to Google Drive to skip ~53 s recompile across sessions
persist_af3_cache_to_drive = False #@param {type:"boolean"}
#@markdown Additional options
force_redownload = False #@param {type:"boolean"}
force_reinstall = False #@param {type:"boolean"}
check_GPU_runtime = True #@param {type:"boolean"}

console = Console()

if force_reinstall:
  os.environ['UV_VENV_CLEAR'] = '1'

# Silence noisy subprocess warnings & unused local-mirror warnings from atomworks
os.environ['PYTHONWARNINGS'] = 'ignore'
os.environ['CCD_MIRROR_PATH'] = '/tmp'
os.environ['PDB_MIRROR_PATH'] = '/tmp'

if check_GPU_runtime:
  try:
    res = subprocess.run(['nvidia-smi', '-L'], capture_output=True, text=True, check=True)
    print(res.stdout.strip(), flush=True)
  except Exception as e:
    print(f"No GPU available ({e}). Will not proceed with setup.", file=sys.stderr, flush=True)
    raise SystemExit(1)

# Model flags
USE_BOLTZ = (folding_model == "Boltz2")
USE_RF3   = (folding_model == "RosettaFold3")
USE_AF3   = (folding_model == "AlphaFold3")
USE_OF3   = (folding_model == "OpenFold3")
USE_AF3_STACK = USE_AF3 or USE_OF3

# Mount Google Drive once on the main thread if any persistence option is enabled
drive_mounted = False
if persist_foundry_checkpoints_to_drive or (USE_AF3_STACK and persist_af3_cache_to_drive):
  try:
    from google.colab import drive
    drive.mount('/content/drive')
    drive_mounted = True
  except Exception as e:
    console.print(f"[yellow][Drive] Mount failed, using local /content storage: {e}[/yellow]")

ENVS_DIR = '/content/envs'
FOUNDRY_CKPTS = (
    '/content/drive/MyDrive/foundry_checkpoints'
    if (persist_foundry_checkpoints_to_drive and drive_mounted)
    else '/content/foundry/checkpoints'
)
AF3_DIR = '/content/af3'
AF3_VERSION = '3.1.18'
AF3_WEIGHTS_URL = 'https://storage.googleapis.com/alphafold3/af3.bin.zst'
AF3_CACHE_DIR = (
    '/content/drive/MyDrive/.af3_cache'
    if (USE_AF3_STACK and persist_af3_cache_to_drive and drive_mounted)
    else os.path.join(tempfile.gettempdir(), 'af3_cache')
)

os.environ.update({
    'FOLDING_MODEL': folding_model,
    'USE_BOLTZ': str(USE_BOLTZ),
    'USE_RF3': str(USE_RF3),
    'USE_AF3': str(USE_AF3),
    'USE_OF3': str(USE_OF3),
    'ENVS_DIR': ENVS_DIR,
    'FOUNDRY_CHECKPOINT_DIRS': FOUNDRY_CKPTS,
})
if USE_AF3_STACK:
  os.makedirs(AF3_CACHE_DIR, exist_ok=True)
  os.environ['AF3_CACHE_DIR'] = AF3_CACHE_DIR

# Bootstrap uv and aria2c
if shutil.which('uv') is None:
  subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'uv'], check=True)
if shutil.which('aria2c') is None:
  subprocess.run(['apt-get', '-qq', 'update'], check=True)
  subprocess.run(['apt-get', '-qq', 'install', '-y', 'aria2'], check=True)

os.makedirs(ENVS_DIR, exist_ok=True)
os.chdir('/content')


def run_command_with_env(env: str, cmd: list[str], quiet=False, check=True):
  """Run a command inside a uv virtual environment and stream live output to Colab."""
  env_path = pathlib.Path(ENVS_DIR) / env
  if not env_path.exists():
    raise FileNotFoundError(f"uv environment '{env}' does not exist at {env_path}")
  if not quiet:
    print(f"[{env}] {' '.join(cmd)}", flush=True)
  sub_env = os.environ.copy()
  sub_env['VIRTUAL_ENV'] = str(env_path)
  sub_env['PATH'] = f"{env_path}/bin:{sub_env['PATH']}"
  sub_env['PYTHONUNBUFFERED'] = '1'

  proc = subprocess.Popen(
      cmd, env=sub_env,
      stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
      text=True, bufsize=1,
  )
  for line in proc.stdout:
    if not quiet:
      print(line, end='', flush=True)
  rc = proc.wait()
  if check and rc != 0:
    raise subprocess.CalledProcessError(rc, cmd)
  return proc


def _download_with_rich(url: str, dest: pathlib.Path, progress: Progress, use_aria2: bool = False, min_bytes: int = 1000):
  """Download a file using a shared rich.progress bar, preserving final speed and elapsed time."""
  dest.parent.mkdir(parents=True, exist_ok=True)
  if dest.exists() and dest.stat().st_size >= min_bytes and not force_redownload:
    return

  t_start = time.time()
  task_id = progress.add_task(f"  Downloading {dest.name}", total=None)

  if use_aria2:
    unit_mult = {'B': 1, 'KiB': 1024, 'MiB': 1024**2, 'GiB': 1024**3, 'TiB': 1024**4}
    prog_re = re.compile(r'(\d+(?:\.\d+)?)(B|KiB|MiB|GiB|TiB)/(\d+(?:\.\d+)?)(B|KiB|MiB|GiB|TiB)')
    cmd = [
        'aria2c', '-x', '16', '-s', '16', '-k', '1M',
        '--allow-overwrite=true', '--summary-interval=1',
        '--download-result=hide', '--console-log-level=notice',
        '-d', str(dest.parent), '-o', dest.name, url,
    ]
    proc = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
    for line in proc.stdout:
      m = prog_re.search(line)
      if m:
        done = int(float(m.group(1)) * unit_mult[m.group(2)])
        total = int(float(m.group(3)) * unit_mult[m.group(4)])
        # Keep task open until aria2c exits so rounded GiB values don't finish it early
        progress.update(task_id, total=total, completed=min(done, max(total - 1, 0)))
    rc = proc.wait()
    if rc != 0 or not dest.exists() or dest.stat().st_size < min_bytes:
      raise RuntimeError(f"aria2c failed for {url}")
    size = dest.stat().st_size
    progress.update(task_id, total=size, completed=size)
    progress._tasks[task_id].finished_speed = size / max(time.time() - t_start, 1e-3)
  else:
    tmp = dest.with_suffix(dest.suffix + '.part')
    def _hook(blocks, block_size, total_size):
      if total_size > 0:
        progress.update(task_id, total=total_size, completed=min(blocks * block_size, max(total_size - 1, 0)))
    urllib.request.urlretrieve(url, tmp, reporthook=_hook)
    if tmp.stat().st_size < min_bytes:
      tmp.unlink(missing_ok=True)
      raise RuntimeError(f"Incomplete download from {url}")
    tmp.replace(dest)
    size = dest.stat().st_size
    progress.update(task_id, total=size, completed=size)
    progress._tasks[task_id].finished_speed = size / max(time.time() - t_start, 1e-3)


def _download_foundry_checkpoints_fast(needed: list[str], ckpt_dir: pathlib.Path, progress: Progress):
  """Resolve Foundry checkpoint URLs and save them under Foundry's canonical filenames via aria2c."""
  canonical_names = {
      'rfd3': 'rfd3_latest.ckpt',
      'rf3': 'rf3_foundry_01_24_latest_remapped.pt',
  }
  lookup = (
      "import json, pathlib, re, foundry; "
      "pkg = pathlib.Path(foundry.__file__).parent; "
      "urls = set(); "
      "[urls.update(re.findall(r'https?://[^\\s\"\\x27<>]+\\.(?:ckpt|pt)', p.read_text(errors='ignore'))) "
      " for p in pkg.rglob('*') if p.suffix in ('.py', '.json', '.yaml', '.yml')]; "
      "print(json.dumps(sorted(urls)))"
  )
  res = subprocess.run(
      [str(pathlib.Path(ENVS_DIR) / 'foundry/bin/python'), '-c', lookup],
      capture_output=True, text=True,
  )
  all_urls = []
  if res.returncode == 0 and res.stdout.strip():
    try:
      all_urls = json.loads(res.stdout.strip().splitlines()[-1])
    except Exception:
      all_urls = []

  fallback = []
  for name in needed:
    matches = [u for u in all_urls if f"/{name}_" in u.lower() or f"/{name}." in u.lower()]
    url = next((u for u in matches if 'latest' in u.lower()), matches[0] if matches else None)
    dest_name = canonical_names.get(name, url.split('/')[-1].split('?')[0] if url else None)
    if url and dest_name:
      dest = ckpt_dir / dest_name
      try:
        _download_with_rich(url, dest, progress, use_aria2=True, min_bytes=10_000_000)
      except Exception:
        fallback.append(name)
    else:
      fallback.append(name)

  if fallback:
    run_command_with_env('foundry', ['foundry', 'install', *fallback, '--checkpoint-dir', str(ckpt_dir)], quiet=True)


# 1. Foundry (RFD3 + built-in MPNN + optional RF3)
def setup_foundry(progress: Progress):
  t0 = time.time()
  env_path = pathlib.Path(ENVS_DIR) / 'foundry'
  ckpt_dir = pathlib.Path(FOUNDRY_CKPTS)

  if not env_path.exists() or force_reinstall:
    extra = 'all' if USE_RF3 else 'rfd3,mpnn'
    subprocess.run(['uv', 'venv', '-q', str(env_path), '--python', '3.12'], check=True)
    subprocess.run(
        ['uv', 'pip', 'install', '-q', '--python', str(env_path),
         f'rc-foundry[{extra}]', 'atomworks[ml]<3', '-U'],
        check=True,
    )
    site_custom = env_path / 'lib/python3.12/site-packages/sitecustomize.py'
    site_custom.write_text("import logging\nlogging.getLogger('atomworks.io').setLevel(logging.ERROR)\n")

  if force_redownload and ckpt_dir.exists():
    shutil.rmtree(ckpt_dir)
  ckpt_dir.mkdir(parents=True, exist_ok=True)

  needed = ['rfd3'] + (['rf3'] if USE_RF3 else [])
  expected = ['rfd3_latest.ckpt'] + (['rf3_foundry_01_24_latest_remapped.pt'] if USE_RF3 else [])
  if not all((ckpt_dir / f).exists() for f in expected):
    _download_foundry_checkpoints_fast(needed, ckpt_dir, progress)

  mpnn_weights = {
      # ProteinMPNN (48 nearest neighbors; σ = 0.02, 0.10, 0.20, 0.30 Å)
      'proteinmpnn_v_48_002.pt': 'https://files.ipd.uw.edu/pub/ligandmpnn/proteinmpnn_v_48_002.pt',
      'proteinmpnn_v_48_010.pt': 'https://files.ipd.uw.edu/pub/ligandmpnn/proteinmpnn_v_48_010.pt',
      'proteinmpnn_v_48_020.pt': 'https://files.ipd.uw.edu/pub/ligandmpnn/proteinmpnn_v_48_020.pt',
      'proteinmpnn_v_48_030.pt': 'https://files.ipd.uw.edu/pub/ligandmpnn/proteinmpnn_v_48_030.pt',
      # LigandMPNN (32 nearest neighbors, 25 ligand atom context; σ = 0.05, 0.10, 0.20, 0.30 Å)
      'ligandmpnn_v_32_005_25.pt': 'https://files.ipd.uw.edu/pub/ligandmpnn/ligandmpnn_v_32_005_25.pt',
      'ligandmpnn_v_32_010_25.pt': 'https://files.ipd.uw.edu/pub/ligandmpnn/ligandmpnn_v_32_010_25.pt',
      'ligandmpnn_v_32_020_25.pt': 'https://files.ipd.uw.edu/pub/ligandmpnn/ligandmpnn_v_32_020_25.pt',
      'ligandmpnn_v_32_030_25.pt': 'https://files.ipd.uw.edu/pub/ligandmpnn/ligandmpnn_v_32_030_25.pt',
      # SolubleMPNN (48 nearest neighbors; σ = 0.02, 0.10, 0.20, 0.30 Å)
      'solublempnn_v_48_002.pt': 'https://files.ipd.uw.edu/pub/ligandmpnn/solublempnn_v_48_002.pt',
      'solublempnn_v_48_010.pt': 'https://files.ipd.uw.edu/pub/ligandmpnn/solublempnn_v_48_010.pt',
      'solublempnn_v_48_020.pt': 'https://files.ipd.uw.edu/pub/ligandmpnn/solublempnn_v_48_020.pt',
      'solublempnn_v_48_030.pt': 'https://files.ipd.uw.edu/pub/ligandmpnn/solublempnn_v_48_030.pt',
  }

  for fname, url in mpnn_weights.items():
    _download_with_rich(url, ckpt_dir / fname, progress, use_aria2=False, min_bytes=100_000)

  return f"[bold green]✓ Foundry ready ({time.time() - t0:.0f} s)[/bold green] — checkpoints in [magenta]{ckpt_dir}[/magenta]"


# 2. Boltz-2 (optional)
def setup_boltz(progress: Progress):
  t0 = time.time()
  env_path = pathlib.Path(ENVS_DIR) / 'boltz'
  boltz_cache = pathlib.Path('/content/boltz/cache')
  os.environ['BOLTZ_CACHE'] = str(boltz_cache)

  if not env_path.exists() or force_reinstall:
    subprocess.run(['uv', 'venv', '-q', str(env_path), '--python', '3.10'], check=True)
    subprocess.run(['uv', 'pip', 'install', '-q', '--python', str(env_path), 'boltz[cuda]', '-U'], check=True)

  if force_redownload and boltz_cache.exists():
    shutil.rmtree(boltz_cache)
  boltz_cache.mkdir(parents=True, exist_ok=True)

  expected = ['mols.tar', 'boltz2_conf.ckpt', 'boltz2_aff.ckpt']
  if not all((boltz_cache / f).exists() for f in expected):
    dummy = pathlib.Path('/tmp/_boltz_prefetch.fasta')
    dummy.write_text(">A|protein|empty\nMAA\n")
    run_command_with_env(
        'boltz',
        ['boltz', 'predict', str(dummy), '--cache', str(boltz_cache), '--out_dir', '/tmp/_boltz_tmp'],
        quiet=True, check=False,
    )
    shutil.rmtree('/tmp/_boltz_tmp', ignore_errors=True)
    dummy.unlink(missing_ok=True)

  return f"[bold green]✓ Boltz-2 ready ({time.time() - t0:.0f} s)[/bold green] — cache in [magenta]{boltz_cache}[/magenta]"


# 3. AlphaFold3 / OpenFold3 (optional)
def setup_af3_stack(progress: Progress):
  t0 = time.time()
  af3_dir = pathlib.Path(AF3_DIR)
  af3_dir.mkdir(parents=True, exist_ok=True)
  env_path = pathlib.Path(ENVS_DIR) / 'af3'
  ready_stamp = af3_dir / f'ALPHAFOLD3_READY_{AF3_VERSION}'

  af3_model = 'alphafold3' if USE_AF3 else 'openfold3'
  af3_precision = 'fp32' if USE_AF3 else 'int8'
  af3_weights_dir = af3_dir / ('af3_native_weights' if USE_AF3 else 'weights')
  weights_stamp = af3_dir / f'WEIGHTS_DONE_{af3_model}_{af3_precision}'

  os.environ.update({
      'AF3_DIR': str(af3_dir),
      'AF3_MODEL': af3_model,
      'AF3_PRECISION': af3_precision,
      'AF3_WEIGHTS_DIR': str(af3_weights_dir),
  })

  if not env_path.exists() or not ready_stamp.exists() or force_reinstall:
    subprocess.run(['uv', 'venv', '-q', '--system-site-packages', str(env_path), '--python', sys.executable], check=True)
    colabfold_src = os.environ.get('RGI_COLABFOLD_SOURCE', 'git+https://github.com/sokrypton/ColabFold')
    subprocess.run(['uv', 'pip', 'install', '-q', '--python', str(env_path), '--no-deps', colabfold_src], check=True)
    subprocess.run(
        ['uv', 'pip', 'install', '-q', '--python', str(env_path),
         'dm-haiku==0.0.17', 'rdkit==2025.9.4', 'tokamax==0.0.11', 'ml_collections',
         'git+https://github.com/sokrypton/py2Dmol.git'],
        check=True,
    )
    subprocess.run(['uv', 'pip', 'uninstall', '-q', '--python', str(env_path), 'jax', 'jaxlib'], check=False)

    for attempt in range(4):
      res = subprocess.run(
          ['uv', 'pip', 'install', '-q', '--python', str(env_path), '--no-deps',
           f'alphafold3-colabfold=={AF3_VERSION}']
      )
      if res.returncode == 0:
        break
      time.sleep(20)
    else:
      raise RuntimeError(f"Could not install alphafold3-colabfold=={AF3_VERSION}")

    _download_with_rich(
        f'https://raw.githubusercontent.com/sokrypton/alphafold3/v{AF3_VERSION}/run_alphafold.py',
        af3_dir / 'run_alphafold.py',
        progress,
    )

    init_code = (
        "import os, pathlib, importlib.metadata as md, haiku, tokamax, alphafold3; "
        "from alphafold3.constants import ccd_fetch; "
        "jp = pathlib.Path(haiku.__file__).parent / '_src' / 'jaxpr_info.py'; "
        "jp.write_text(jp.read_text().replace('jax.core.DropVar', 'jax.extend.core.DropVar')) "
        "if jp.exists() and 'jax.core.DropVar' in jp.read_text() else None; "
        "gu = pathlib.Path(tokamax.__file__).parent / '_src' / 'gpu_utils.py'; "
        "old = 'return float(device.compute_capability) >= 8.0'; "
        "new = 'cc = float(device.compute_capability)\\n  return cc == 8.0 or cc >= 9.0'; "
        "gu.write_text(gu.read_text().replace(old, new)) if gu.exists() and old in gu.read_text() else None; "
        "root = os.path.dirname(md.distribution('alphafold3-colabfold').locate_file('alphafold3')); "
        "conv = os.path.join(root, 'alphafold3', 'constants', 'converters'); "
        "os.makedirs(conv, exist_ok=True); "
        "ccd_fetch.write_pickles(ccd_fetch.codes_for_input(), os.path.join(conv, 'ccd.pickle'), "
        "os.path.join(conv, 'chemical_component_sets.pickle'), libcifpp_dir=os.path.join(root, 'share', 'libcifpp'))"
    )
    run_command_with_env('af3', ['python', '-c', init_code], quiet=True)
    ready_stamp.touch()

  if force_redownload:
    shutil.rmtree(af3_weights_dir, ignore_errors=True)
    weights_stamp.unlink(missing_ok=True)
  af3_weights_dir.mkdir(parents=True, exist_ok=True)

  if USE_AF3:
    af3_bin = af3_weights_dir / 'af3.bin.zst'
    if not glob.glob(str(af3_weights_dir / '*.bin.zst')):
      _download_with_rich(AF3_WEIGHTS_URL, af3_bin, progress, use_aria2=True, min_bytes=1_000_000)
    weights_stamp.touch()
  elif not weights_stamp.exists():
    run_command_with_env(
        'af3',
        ['python', '-c', f"from alphafold3.model import weights; weights.ensure_weights('{af3_model}', None, precision='{af3_precision}')"],
        quiet=True,
    )
    weights_stamp.touch()

  return f"[bold green]✓ {folding_model} ready ({time.time() - t0:.0f} s)[/bold green] — weights in [magenta]{af3_weights_dir}[/magenta]"


# Header -> Progress bars -> Summary -> GPU test (clean top-to-bottom layout)
tasks = [setup_foundry]
if USE_BOLTZ:
  tasks.append(setup_boltz)
if USE_AF3_STACK:
  tasks.append(setup_af3_stack)

console.print(f"[bold blue]Installing Foundry[/bold blue] (RFD3 + MPNN) and [bold blue]{folding_model}[/bold blue] in parallel...")

summary = []
with Progress(
    TextColumn("{task.description}"),
    BarColumn(),
    DownloadColumn(),
    TransferSpeedColumn(),
    TimeRemainingColumn(elapsed_when_finished=True),
    console=console,
) as progress:
  with ThreadPoolExecutor(max_workers=len(tasks)) as pool:
    futures = [pool.submit(fn, progress) for fn in tasks]
    for fut in as_completed(futures):
      summary.append(fut.result())

for line in summary:
  if line:
    console.print(line)

# Verify GPU access across active environments
console.print("\n[bold]========== TEST GPU-CUDA ==========[/bold]")
torch_test = "import torch; print('PyTorch:', torch.__version__, 'CUDA:', torch.cuda.is_available())"
run_command_with_env("foundry", ["python", "-c", torch_test])
if USE_BOLTZ:
  run_command_with_env("boltz", ["python", "-c", torch_test])
if USE_AF3_STACK:
  jax_test = "import jax, alphafold3; print('JAX:', jax.__version__, 'Devices:', jax.devices())"
  run_command_with_env("af3", ["python", "-c", jax_test])

# ===========================
#@markdown VARS that are set and used later
#@markdown | Variable | Value | Set |
#@markdown |---|---|---|
#@markdown | `FOLDING_MODEL` | `"RosettaFold3"` / `"Boltz2"` / `"AlphaFold3"` / `"OpenFold3"` | always |
#@markdown | `USE_BOLTZ`, `USE_RF3`, `USE_AF3`, `USE_OF3` | `"True"` / `"False"` (string) | always |
#@markdown | `ENVS_DIR` | `/content/envs` | always |
#@markdown | `FOUNDRY_CHECKPOINT_DIRS` | `/content/foundry/checkpoints` or `/content/drive/MyDrive/foundry_checkpoints` | always |
#@markdown | `BOLTZ_CACHE` | `/content/boltz/cache` | only if `USE_BOLTZ` |
#@markdown | `AF3_DIR` | `/content/af3` (contains `run_alphafold.py`) | only if `USE_AF3` or `USE_OF3` |
#@markdown | `AF3_MODEL` | `"alphafold3"` / `"openfold3"` | only if `USE_AF3` or `USE_OF3` |
#@markdown | `AF3_PRECISION` | `"fp32"` (AF3) / `"int8"` (OpenFold3) | only if `USE_AF3` or `USE_OF3` |
#@markdown | `AF3_WEIGHTS_DIR` | `/content/af3/af3_native_weights` or `/content/af3/weights` | only if `USE_AF3` or `USE_OF3` |
#@markdown | `AF3_CACHE_DIR` | `/tmp/af3_cache` or `/content/drive/MyDrive/.af3_cache` | only if `USE_AF3` or `USE_OF3` |

In [ ]:
#@title Helper functions to compare symmetric RFD3 designs vs structure predictions

try:
  import biotite
except ModuleNotFoundError:
  !pip -q install biotite

import glob, gzip, io, itertools, json, pathlib, uuid
import numpy as np
import biotite.structure as struc
import biotite.structure.io.pdb as pdb_io
import biotite.structure.io.pdbx as pdbx_io
from IPython.display import display, HTML

COLORS = ["#4C78A8", "#F58518", "#54A24B", "#E45756", "#B279A2", "#72B7B2", "#EECA3B", "#FF9DA6"]
CHAIN_LABELS = list("ABCDEFGHIJKLMNOPQRSTUVWXYZabcdefghijklmnopqrstuvwxyz0123456789")


def _strip_struct_ext(path: pathlib.Path) -> str:
  name = path.name
  for ext in (".gz", ".cif", ".pdb", ".bcif"):
    name = name.removesuffix(ext)
  return name


def _load(path):
  p = str(path)
  opener = gzip.open if p.endswith(".gz") else open
  base = p[:-3] if p.endswith(".gz") else p
  if base.endswith(".bcif"):
    with opener(p, "rb") as f:
      return pdbx_io.get_structure(pdbx_io.BinaryCIFFile.read(f), model=1, extra_fields=["b_factor"])
  if base.endswith(".cif"):
    with opener(p, "rt") as f:
      return pdbx_io.get_structure(pdbx_io.CIFFile.read(f), model=1, extra_fields=["b_factor"])
  with opener(p, "rt") as f:
    return pdb_io.PDBFile.read(f).get_structure(model=1, extra_fields=["b_factor"])


def _to_pdb(atoms):
  atoms = atoms[struc.filter_amino_acids(atoms)].copy()
  # Map unique chain IDs (including multi-char IDs like A0/A1) to single-char PDB chain IDs
  uniq_cids = list(dict.fromkeys(atoms.chain_id.astype(str)))
  cid_map = {c: CHAIN_LABELS[i % len(CHAIN_LABELS)] for i, c in enumerate(uniq_cids)}
  atoms.chain_id = np.array([cid_map[c] for c in atoms.chain_id.astype(str)])
  buf = io.StringIO()
  f = pdb_io.PDBFile()
  f.set_structure(atoms)
  f.write(buf)
  return buf.getvalue()


def _kabsch(P, Q):
  """Rigidly align mobile coordinates Q onto fixed coordinates P; returns (R, t, rmsd)."""
  p_mean = P.mean(axis=0)
  q_mean = Q.mean(axis=0)
  H = (Q - q_mean).T @ (P - p_mean)
  U, _, Vt = np.linalg.svd(H)
  d = np.sign(np.linalg.det(Vt.T @ U.T))
  R = Vt.T @ np.diag([1.0, 1.0, d]) @ U.T
  t = p_mean - q_mean @ R.T
  Q_fit = Q @ R.T + t
  rmsd = float(np.sqrt(np.mean(np.sum((P - Q_fit) ** 2, axis=-1))))
  return R, t, rmsd


def _chain_ca_coords(struct):
  """Return list of Cα coordinate arrays, one per protein chain (in appearance order)."""
  ca = struct[(struct.atom_name == "CA") & struc.filter_amino_acids(struct)]
  cids = list(dict.fromkeys(ca.chain_id.astype(str)))
  return [ca.coord[ca.chain_id.astype(str) == cid] for cid in cids]


def _align_oligomer(fixed, mobile):
  """Permutation-invariant Cα alignment for homo-oligomers (returns aligned mobile, complex RMSD, monomer RMSD, note)."""
  f_chains = _chain_ca_coords(fixed)
  m_chains = _chain_ca_coords(mobile)
  n_chains = len(f_chains)

  # Multi-chain homo-oligomer with matching chain count and lengths
  if (
      n_chains >= 2
      and len(m_chains) == n_chains
      and len({len(c) for c in f_chains + m_chains}) == 1
  ):
    L = len(f_chains[0])
    P_all = np.vstack(f_chains)

    if n_chains <= 4:
      # Exhaustive search (<= 24 permutations)
      perms = list(itertools.permutations(range(n_chains)))
    else:
      # Anchor mobile chain 0 to each fixed chain k, then greedily match remaining chains by centroid
      f_cents = np.array([c.mean(axis=0) for c in f_chains])
      m_cents = np.array([c.mean(axis=0) for c in m_chains])
      perms = []
      for k in range(n_chains):
        R0, t0, _ = _kabsch(f_chains[k], m_chains[0])
        mc_aligned = m_cents @ R0.T + t0
        used, perm = set(), [None] * n_chains
        for fi in range(n_chains):
          dists = np.linalg.norm(mc_aligned - f_cents[fi], axis=-1)
          for mi in np.argsort(dists):
            if mi not in used:
              perm[fi] = int(mi)
              used.add(int(mi))
              break
        perms.append(tuple(perm))

    best_R, best_t, best_rmsd, best_perm = None, None, np.inf, None
    for perm in perms:
      Q_all = np.vstack([m_chains[i] for i in perm])
      R, t, rmsd = _kabsch(P_all, Q_all)
      if rmsd < best_rmsd:
        best_R, best_t, best_rmsd, best_perm = R, t, rmsd, perm

    aligned = mobile.copy()
    aligned.coord = mobile.coord @ best_R.T + best_t

    # Mean single-subunit (monomer) Cα RMSD across matched chains
    mono_rmsds = [_kabsch(f_chains[fi], m_chains[mi])[2] for fi, mi in enumerate(best_perm)]
    mono_rmsd = float(np.mean(mono_rmsds))
    note = f"complex {best_rmsd:.2f} Å | monomer {mono_rmsd:.2f} Å ({n_chains}×{L} res)"
    return aligned, best_rmsd, mono_rmsd, note

  # Fallback: single chain or mismatched lengths
  f_ca = fixed[(fixed.atom_name == "CA") & struc.filter_amino_acids(fixed)]
  m_ca = mobile[(mobile.atom_name == "CA") & struc.filter_amino_acids(mobile)]
  if len(f_ca) == len(m_ca) and len(f_ca) >= 3:
    R, t, rmsd = _kabsch(f_ca.coord, m_ca.coord)
    aligned = mobile.copy()
    aligned.coord = mobile.coord @ R.T + t
    return aligned, rmsd, rmsd, f"{rmsd:.2f} Å ({len(f_ca)} res)"

  res = struc.superimpose_structural_homologs(fixed, mobile)
  tf, f_idx, m_idx = res[1:] if len(res) == 4 else res
  rmsd = float(struc.rmsd(fixed[f_idx], tf.apply(mobile[m_idx])))
  return tf.apply(mobile), rmsd, rmsd, f"{rmsd:.2f} Å ({len(f_idx)} aligned res)"


def browse_designs(
    design_glob,
    pred_root,
    only_best=True,
    color_by_chain=True,
    max_models=5,
    width=700,
    height=500,
):
  """Interactive viewer to compare symmetric RFD3 oligomers against AF3 / OpenFold3 / Boltz-2 / RF3 models."""
  pred_root = pathlib.Path(pred_root)
  payload = []

  for design_path in sorted(glob.glob(str(design_glob))):
    stem = _strip_struct_ext(pathlib.Path(design_path))

    candidates = (
        pred_root / stem, pred_root / stem.lower(),
        pred_root / f"{stem}.cif", pred_root / f"{stem}.cif".lower(),
        pred_root / "predictions" / stem, pred_root / "predictions" / f"{stem}.cif",
    )
    design_dir = next((d for d in candidates if d.is_dir()), None)
    if design_dir is not None:
      matches = [
          p for pat in ("*.cif", "*.cif.gz", "*.pdb")
          for p in sorted(design_dir.rglob(pat))
          if p.is_file() and "inputs" not in p.parts
      ]
    else:
      matches = [
          p for pat in (f"{stem}*model*.cif", f"{stem}*model*.cif.gz", f"{stem}*model*.pdb")
          for p in sorted(pred_root.rglob(pat))
          if p.is_file() and "inputs" not in p.parts
      ]

    if only_best:
      top_dir = design_dir if design_dir is not None else pred_root
      top_af3_rf3 = [m for m in matches if m.parent == top_dir and m.name.endswith(("_model.cif", "_model.cif.gz"))]
      top_boltz = [m for m in matches if m.name.endswith(("_model_0.cif", "_model_0.pdb"))]
      matches = (top_af3_rf3 or top_boltz or matches)[:1]
    else:
      samples = [m for m in matches if "sample-" in str(m) or "_model_" in m.name]
      matches = (samples or matches)[:max_models]

    if not matches:
      continue

    fixed = _load(design_path)
    models = []
    for i, m in enumerate(matches):
      aligned, _, _, note = _align_oligomer(fixed, _load(m))
      models.append({
          "label": m.parent.name if m.name == "model.cif" else _strip_struct_ext(m),
          "color": COLORS[i % len(COLORS)],
          "pdb": _to_pdb(aligned),
          "note": note,
      })
    payload.append({"stem": stem, "fixed_pdb": _to_pdb(fixed), "models": models})

  if not payload:
    raise FileNotFoundError("No matching design/prediction pairs found. Check design_glob and pred_root.")

  uid = uuid.uuid4().hex[:8]
  data_json = json.dumps(payload).replace("</", "<\\/")
  palette_json = json.dumps(COLORS)
  per_chain_js = "true" if (color_by_chain and only_best) else "false"

  display(HTML(f"""
  <div style="font-family:sans-serif;max-width:{width}px;">
    <div style="display:flex;gap:6px;align-items:center;margin-bottom:8px;">
      <button id="prev_{uid}" style="padding:5px 12px;cursor:pointer;">◀ Prev</button>
      <select id="sel_{uid}" style="flex:1;padding:5px;"></select>
      <button id="next_{uid}" style="padding:5px 12px;cursor:pointer;">Next ▶</button>
    </div>
    <div id="view_{uid}" style="position:relative;width:{width}px;height:{height}px;border:1px solid #ddd;"></div>
    <div id="leg_{uid}" style="font-size:12px;color:#444;margin-top:6px;line-height:1.6;"></div>
  </div>
  <script>
  (function() {{
    const DATA = {data_json};
    const PALETTE = {palette_json};
    const COLOR_BY_CHAIN = {per_chain_js};
    const sel = document.getElementById("sel_{uid}");
    const leg = document.getElementById("leg_{uid}");
    const esc = s => String(s).replace(/[&<>"']/g, c => ({{"&":"&amp;","<":"&lt;",">":"&gt;",'"':"&quot;","'":"&#39;"}}[c]));
    const swatch = (c, t, m) => `<span style="display:inline-block;width:10px;height:10px;background:${{c}};border:1px solid #888;margin:0 4px 0 ${{m}}px;vertical-align:middle;"></span>${{esc(t)}}`;

    function init() {{
      const v = $3Dmol.createViewer(document.getElementById("view_{uid}"), {{ backgroundColor: "white" }});
      DATA.forEach((d, i) => sel.add(new Option(`${{i + 1}}/${{DATA.length}} — ${{d.stem}} (${{d.models.length}} model${{d.models.length > 1 ? "s" : ""}})`, i)));

      function render() {{
        const d = DATA[Number(sel.value)];
        v.removeAllModels();
        v.addModel(d.fixed_pdb, "pdb");
        v.setStyle({{ model: 0 }}, {{ cartoon: {{ color: "gainsboro", opacity: 0.75 }} }});
        let html = swatch("gainsboro", `${{d.stem}} (RFD3)`, 0);

        d.models.forEach((m, i) => {{
          const mdl = v.addModel(m.pdb, "pdb");
          if (COLOR_BY_CHAIN && d.models.length === 1) {{
            const chains = [...new Set(mdl.selectedAtoms({{}}).map(a => a.chain))];
            chains.forEach((ch, ci) => {{
              v.setStyle({{ model: i + 1, chain: ch }}, {{ cartoon: {{ color: PALETTE[ci % PALETTE.length] }} }});
            }});
            html += `<span style="margin-left:12px;"><b>${{esc(m.label)}}</b> (${{esc(m.note)}})</span>`;
            chains.forEach((ch, ci) => {{ html += swatch(PALETTE[ci % PALETTE.length], `Chain ${{ch}}`, 8); }});
          }} else {{
            v.setStyle({{ model: i + 1 }}, {{ cartoon: {{ color: m.color }} }});
            html += swatch(m.color, `${{m.label}} (${{m.note}})`, 12);
          }}
        }});
        v.zoomTo(); v.render();
        leg.innerHTML = html;
      }}

      sel.onchange = render;
      document.getElementById("prev_{uid}").onclick = () => {{ sel.value = (Number(sel.value) - 1 + DATA.length) % DATA.length; render(); }};
      document.getElementById("next_{uid}").onclick = () => {{ sel.value = (Number(sel.value) + 1) % DATA.length; render(); }};
      render();
    }}

    if (window.$3Dmol) init();
    else {{ const s = document.createElement("script"); s.src = "https://3Dmol.org/build/3Dmol-min.js"; s.onload = init; document.head.appendChild(s); }}
  }})();
  </script>
  """))

# Usage:
# browse_designs("unconditional_symmetry_test_example/rfd3_out/*.cif.gz", "unconditional_symmetry_test_example/alphafold3_out", only_best=True)
# browse_designs("unconditional_symmetry_test_example/rfd3_out/*.cif.gz", "unconditional_symmetry_test_example/boltz_out", only_best=True)

In [ ]:
#@title Run RFDiffusion3 — Unconditional Symmetry Design
#@markdown Generates *de novo* symmetric homo-oligomers (**Cyclic `Cn`** or **Dihedral `Dn`**).

#@markdown ### Execution Mode
execution_mode = "Run in Colab" #@param ["Run in Colab", "Prepare inputs & print terminal commands"] {type:"string"}

#@markdown ### Output & Symmetry
job_name = "test_symmetry" #@param {type:"string"}
out_dir = "unconditional_symmetry_test_example/rfd3_out" #@param {type:"string"}
#@markdown Pick a preset or type any valid `Cn` (*n* subunits) or `Dn` (2*n* subunits) symmetry (e.g., `C8`, `D7`).
symmetry_id = "C3" #@param ["C2", "C3", "C4", "C5", "C6", "D2", "D3", "D4", "D5", "D6"] {allow-input: true}
#@markdown **Per-subunit** length: single value (`"80"`), random range per batch (`"60-90"`), or comma-separated sizes (`"60, 80, 100"`).
subunit_lengths = "60-90" #@param {type:"string"}
#@markdown Total designs per size entry = `n_batches × diffusion_batch_size`. Keep `diffusion_batch_size=1` for symmetry to avoid GPU OOM.
n_batches = 5 #@param {type:"integer"}
diffusion_batch_size = 1 #@param {type:"integer"}

#@markdown ### Designability vs. Diversity (Sampling)
#@markdown - **Balanced (default):** `step_scale=1.5`, `gamma_0=0.6`, `is_non_loopy=True`
#@markdown - **High designability (low-T):** `step_scale=3.0`, `gamma_0=0.2`, `is_non_loopy=True`
#@markdown - **High diversity:** `step_scale=1.0`, `gamma_0=0.8`, `is_non_loopy=False`
step_scale = 1.5 #@param {type:"string"}
gamma_0 = 0.6 #@param {type:"string"}
num_timesteps = 200 #@param {type:"integer"}
#@markdown Biases away from excessive unstructured loops (recommended `True` for higher refolding pass rates)
is_non_loopy = True #@param {type:"boolean"}

#@markdown ### Execution & Memory
#@markdown Enable `low_memory_mode` if total complex length (`n_subunits × subunit_length`) exceeds ~400 aa on a T4 GPU
low_memory_mode = False #@param {type:"boolean"}
dump_trajectories = False #@param {type:"boolean"}
skip_existing = False #@param {type:"boolean"}

import json, os, pathlib, re, shlex

# 1. Validate symmetry ID (RFD3 supports Cn and Dn with n >= 2)
sym_id = symmetry_id.strip().upper()
m_sym = re.fullmatch(r"([CD])(\d+)", sym_id)
if not m_sym or int(m_sym.group(2)) < 2:
  raise ValueError(
      f"Unsupported symmetry '{symmetry_id}'. RFD3 currently supports Cyclic (C2, C3, ...) and Dihedral (D2, D3, ...) symmetries."
  )
sym_type, sym_order = m_sym.group(1), int(m_sym.group(2))
n_subunits = sym_order if sym_type == "C" else 2 * sym_order

# 2. Build RFD3 symmetry specs
length_entries = [x.strip() for x in subunit_lengths.split(",") if x.strip()]
if not length_entries:
  raise ValueError("Specify at least one subunit length (e.g., '80' or '60-90').")

specs = {}
max_subunit_len = 0
for idx, entry in enumerate(length_entries, start=1):
  nums = [int(n) for n in re.findall(r"\d+", entry)]
  if not nums:
    raise ValueError(f"Invalid length entry: '{entry}'")
  max_subunit_len = max(max_subunit_len, max(nums))

  key = f"{job_name}_{sym_id}" if len(length_entries) == 1 else f"{job_name}_{sym_id}_{idx}"
  specs[key] = {
      "length": int(entry) if entry.isdigit() else entry,
      "is_non_loopy": is_non_loopy,
      "symmetry": {"id": sym_id},
  }

rfd3_out = pathlib.Path(out_dir).resolve()
rfd3_out.mkdir(parents=True, exist_ok=True)

# Store symmetry metadata for downstream MPNN (tied positions) and refolding cells
SYMMETRY_STATE = {
    "symmetry_id": sym_id,
    "n_subunits": n_subunits,
    "rfd3_out": str(rfd3_out),
}

config_path = rfd3_out.parent / f"{job_name}_{sym_id}.json"
config_path.write_text(json.dumps(specs, indent=2) + "\n")

total_designs = len(specs) * n_batches * diffusion_batch_size
max_total_res = n_subunits * max_subunit_len
print(
    f"Config saved to {config_path} | Symmetry: {sym_id} ({n_subunits} subunits, up to {max_total_res} total aa) | "
    f"{len(specs)} spec(s) × {n_batches} batch(es) × {diffusion_batch_size} = {total_designs} designs",
    flush=True,
)
print(json.dumps(specs, indent=2))

if len(length_entries) == 1 and "-" in length_entries[0] and n_batches == 1:
  print("Note: n_batches=1 with a length range samples only 1 length for the whole batch. Increase n_batches for multiple sizes.")
if max_total_res > 400 and not low_memory_mode:
  print(f"Note: Complex size is up to {max_total_res} aa. If you hit CUDA OOM, enable low_memory_mode=True.")

# 3. Build RFD3 symmetry command and either print terminal instructions or run
cmd = [
    "rfd3", "design",
    f"inputs={config_path.resolve()}",
    f"out_dir={rfd3_out}",
    'global_prefix=""',
    "inference_sampler.kind=symmetry",
    f"n_batches={n_batches}",
    f"diffusion_batch_size={diffusion_batch_size}",
    f"inference_sampler.num_timesteps={num_timesteps}",
    f"inference_sampler.step_scale={step_scale}",
    f"inference_sampler.gamma_0={gamma_0}",
    f"low_memory_mode={low_memory_mode}",
    f"dump_trajectories={dump_trajectories}",
    f"skip_existing={skip_existing}",
    "prevalidate_inputs=True",
]

if execution_mode == "Prepare inputs & print terminal commands":
  env_path = pathlib.Path(os.environ["ENVS_DIR"]) / "foundry"
  cmd_str = " \\\n  ".join(shlex.quote(c) for c in cmd)
  script = f"""#!/usr/bin/env bash
set -euo pipefail

# Activate Foundry environment & set required paths
export VIRTUAL_ENV={shlex.quote(str(env_path))}
export PATH="{env_path}/bin:$PATH"
export FOUNDRY_CHECKPOINT_DIRS={shlex.quote(os.environ['FOUNDRY_CHECKPOINT_DIRS'])}
export CCD_MIRROR_PATH=/tmp
export PDB_MIRROR_PATH=/tmp
export PYTHONWARNINGS=ignore

# Run RFdiffusion3 (symmetry mode: {sym_id})
{cmd_str}
"""
  sh_path = rfd3_out.parent / f"run_rfd3_{job_name}_{sym_id}.sh"
  sh_path.write_text(script)
  sh_path.chmod(0o755)
  print("\n" + "=" * 72)
  print("TERMINAL INSTRUCTIONS — Copy & paste into a terminal, or run:")
  print(f"  bash {sh_path}")
  print("=" * 72)
  print(script)
else:
  run_command_with_env("foundry", cmd)

In [ ]:
#@title Run MPNN — Symmetric Inverse Folding (via Foundry)
#@markdown Designs tied homo-oligomeric sequences directly on symmetric RFD3 `.cif` / `.cif.gz` backbones.

#@markdown ### Execution Mode
execution_mode = "Run in Colab" #@param ["Run in Colab", "Prepare inputs & print terminal commands"] {type:"string"}

#@markdown ### Input & Output
input_dir = "unconditional_symmetry_test_example/rfd3_out" #@param {type:"string"}
out_dir = "unconditional_symmetry_test_example/mpnn_out" #@param {type:"string"}
#@markdown Tie equivalent positions across all subunits so every chain receives the exact same sequence (`True` for homo-oligomers).
tie_symmetric_chains = True #@param {type:"boolean"}

#@markdown ### Model & Sampling
mpnn_model = "SolubleMPNN (solublempnn_v_48_020.pt)" #@param ["SolubleMPNN (solublempnn_v_48_002.pt)", "SolubleMPNN (solublempnn_v_48_010.pt)", "SolubleMPNN (solublempnn_v_48_020.pt)", "SolubleMPNN (solublempnn_v_48_030.pt)", "LigandMPNN (ligandmpnn_v_32_005_25.pt)", "LigandMPNN (ligandmpnn_v_32_010_25.pt)", "LigandMPNN (ligandmpnn_v_32_020_25.pt)", "LigandMPNN (ligandmpnn_v_32_030_25.pt)", "ProteinMPNN (proteinmpnn_v_48_002.pt)", "ProteinMPNN (proteinmpnn_v_48_010.pt)", "ProteinMPNN (proteinmpnn_v_48_020.pt)", "ProteinMPNN (proteinmpnn_v_48_030.pt)"] {type:"string"}
#@markdown Total sequences per backbone = `batch_size × number_of_batches`
batch_size = 2 #@param {type:"integer"}
number_of_batches = 1 #@param {type:"integer"}
temperature = "0.2" #@param {type:"string"}
seed = 0 #@param {type:"integer"}
#@markdown Comma-separated 3-letter residues to omit (e.g., `CYS`), or leave blank
omit_residues = "" #@param {type:"string"}

import gzip, json, os, pathlib, re, shlex
import biotite.structure as struc
import biotite.structure.io.pdbx as pdbx_io

MPNN_MODELS = {
    "SolubleMPNN (solublempnn_v_48_002.pt)": ("solublempnn_v_48_002.pt", "protein_mpnn"),
    "SolubleMPNN (solublempnn_v_48_010.pt)": ("solublempnn_v_48_010.pt", "protein_mpnn"),
    "SolubleMPNN (solublempnn_v_48_020.pt)": ("solublempnn_v_48_020.pt", "protein_mpnn"),
    "SolubleMPNN (solublempnn_v_48_030.pt)": ("solublempnn_v_48_030.pt", "protein_mpnn"),
    "LigandMPNN (ligandmpnn_v_32_005_25.pt)": ("ligandmpnn_v_32_005_25.pt", "ligand_mpnn"),
    "LigandMPNN (ligandmpnn_v_32_010_25.pt)": ("ligandmpnn_v_32_010_25.pt", "ligand_mpnn"),
    "LigandMPNN (ligandmpnn_v_32_020_25.pt)": ("ligandmpnn_v_32_020_25.pt", "ligand_mpnn"),
    "LigandMPNN (ligandmpnn_v_32_030_25.pt)": ("ligandmpnn_v_32_030_25.pt", "ligand_mpnn"),
    "ProteinMPNN (proteinmpnn_v_48_002.pt)": ("proteinmpnn_v_48_002.pt", "protein_mpnn"),
    "ProteinMPNN (proteinmpnn_v_48_010.pt)": ("proteinmpnn_v_48_010.pt", "protein_mpnn"),
    "ProteinMPNN (proteinmpnn_v_48_020.pt)": ("proteinmpnn_v_48_020.pt", "protein_mpnn"),
    "ProteinMPNN (proteinmpnn_v_48_030.pt)": ("proteinmpnn_v_48_030.pt", "protein_mpnn"),
}

ckpt_name, model_type = MPNN_MODELS[mpnn_model]
ckpt_path = pathlib.Path(os.environ["FOUNDRY_CHECKPOINT_DIRS"]) / ckpt_name
if not ckpt_path.exists():
  raise FileNotFoundError(f"Checkpoint not found: {ckpt_path}. Re-run the Setup cell.")

rfd3_dir = pathlib.Path(input_dir).resolve()
cif_files = sorted(rfd3_dir.glob("*.cif*")) if rfd3_dir.exists() else []

ligandmpnn_out = pathlib.Path(out_dir).resolve()
ligandmpnn_out.mkdir(parents=True, exist_ok=True)

def _load_chain_residues(cif_path: pathlib.Path) -> list[tuple[str, list[int]]]:
  """Return [(chain_id, [res_id, ...]), ...] for all protein chains in a .cif / .cif.gz."""
  opener = gzip.open if cif_path.suffix == ".gz" else open
  with opener(cif_path, "rt") as f:
    atoms = pdbx_io.get_structure(pdbx_io.CIFFile.read(f), model=1)
  ca = atoms[(atoms.atom_name == "CA") & struc.filter_amino_acids(atoms)]
  chains = []
  for cid in dict.fromkeys(ca.chain_id.astype(str)):
    res_ids = [int(r) for r in ca.res_id[ca.chain_id.astype(str) == cid]]
    chains.append((cid, res_ids))
  return chains

omit_list = [r.strip().upper() for r in omit_residues.split(",") if r.strip()]

if execution_mode == "Prepare inputs & print terminal commands":
  env_path = pathlib.Path(os.environ["ENVS_DIR"]) / "foundry"

  # Determine chain IDs for --homo_oligomer_chains (from existing .cif* or SYMMETRY_STATE)
  inferred_chains = []
  if tie_symmetric_chains:
    if cif_files:
      inferred_chains = [cid for cid, _ in _load_chain_residues(cif_files[0])]
    elif "SYMMETRY_STATE" in globals() and SYMMETRY_STATE.get("n_subunits"):
      inferred_chains = list("ABCDEFGHIJKLMNOPQRSTUVWXYZ")[: int(SYMMETRY_STATE["n_subunits"])]

  extra_args = [
      f"--out_directory {shlex.quote(str(ligandmpnn_out))}",
      f"--checkpoint_path {shlex.quote(str(ckpt_path))}",
      f"--model_type {shlex.quote(model_type)}",
      "--is_legacy_weights True",
      f"--batch_size {int(batch_size)}",
      f"--number_of_batches {int(number_of_batches)}",
      f"--temperature {shlex.quote(str(temperature))}",
      f"--seed {int(seed)}",
  ]
  if omit_list:
    extra_args.append(f"--omit {shlex.quote(json.dumps(omit_list))}")
  if tie_symmetric_chains and len(inferred_chains) >= 2:
    extra_args.append(f"--homo_oligomer_chains {shlex.quote(json.dumps([inferred_chains]))}")
  args_block = " \\\n    ".join(extra_args)

  script = f"""#!/usr/bin/env bash
set -euo pipefail

# Activate Foundry environment & set required paths
export VIRTUAL_ENV={shlex.quote(str(env_path))}
export PATH="{env_path}/bin:$PATH"
export FOUNDRY_CHECKPOINT_DIRS={shlex.quote(os.environ['FOUNDRY_CHECKPOINT_DIRS'])}
export CCD_MIRROR_PATH=/tmp
export PDB_MIRROR_PATH=/tmp
export PYTHONWARNINGS=ignore

# Run MPNN over all symmetric RFD3 .cif / .cif.gz backbones
mkdir -p {shlex.quote(str(ligandmpnn_out))}
shopt -s nullglob
for cif in {shlex.quote(str(rfd3_dir))}/*.cif*; do
  echo "Designing sequences for: $cif"
  mpnn \\
    --structure_path "$cif" \\
    {args_block}
done
"""
  sh_path = ligandmpnn_out.parent / "run_mpnn.sh"
  sh_path.write_text(script)
  sh_path.chmod(0o755)
  print(f"Found {len(cif_files)} backbone(s) currently in {rfd3_dir} (loop will process all *.cif* present when run).")
  if tie_symmetric_chains and len(inferred_chains) < 2:
    print("Note: Could not infer subunit chain IDs yet. Run Step 1 first or add --homo_oligomer_chains '[[\"A\",\"B\",...]]' manually.")
  print("\n" + "=" * 72)
  print("TERMINAL INSTRUCTIONS — Copy & paste into a terminal, or run:")
  print(f"  bash {sh_path}")
  print("=" * 72)
  print(script)
else:
  if not cif_files:
    raise FileNotFoundError(f"No .cif or .cif.gz files found in {rfd3_dir}")
  print(f"Running {mpnn_model} on {len(cif_files)} symmetric backbone(s) -> {ligandmpnn_out}", flush=True)

  chain_counts = {}
  for cif_path in cif_files:
    chains = _load_chain_residues(cif_path)
    chain_ids = [cid for cid, _ in chains]
    chain_counts[cif_path.name] = (len(chains), len(chains[0][1]) if chains else 0)

    cmd = [
        "mpnn",
        "--structure_path", str(cif_path.resolve()),
        "--out_directory", str(ligandmpnn_out),
        "--checkpoint_path", str(ckpt_path),
        "--model_type", model_type,
        "--is_legacy_weights", "True",
        "--batch_size", str(batch_size),
        "--number_of_batches", str(number_of_batches),
        "--temperature", str(temperature),
        "--seed", str(seed),
    ]
    if omit_list:
      cmd += ["--omit", json.dumps(omit_list)]
    if tie_symmetric_chains and len(chain_ids) >= 2:
      cmd += ["--homo_oligomer_chains", json.dumps([chain_ids])]

    run_command_with_env("foundry", cmd)

  # Quick sanity check: verify that designed subunits in the first FASTA are identical
  fastas = sorted((ligandmpnn_out / "seqs").glob("*.fa*")) if (ligandmpnn_out / "seqs").is_dir() else sorted(ligandmpnn_out.rglob("*.fa*"))
  if fastas:
    lines = [l.strip() for l in fastas[0].read_text().splitlines() if l.strip()]
    seqs = [l for l in lines if not l.startswith(">")]
    if seqs:
      first_seq = seqs[0]
      n_ch, sub_len = next(iter(chain_counts.values()))
      parts = re.split(r"[:/]", first_seq) if (":" in first_seq or "/" in first_seq) else [
          first_seq[i * sub_len : (i + 1) * sub_len] for i in range(n_ch)
      ] if len(first_seq) == n_ch * sub_len else [first_seq]
      is_tied = len(parts) == n_ch and len(set(parts)) == 1
      status = "IDENTICAL across all subunits" if is_tied else f"NOT identical across subunits ({len(set(parts))} unique)"
      print(f"\nVerification ({fastas[0].name}): {n_ch} chains × {sub_len} aa -> {status}", flush=True)

In [ ]:
#@title Run Refolding — Symmetric Oligomers (AlphaFold3 / OpenFold3 / Boltz-2 / RosettaFold3)
#@markdown Single-sequence multi-chain refolding of symmetric MPNN designs (supports both tied and untied subunit sequences).

#@markdown ### Execution Mode
execution_mode = "Run in Colab" #@param ["Run in Colab", "Prepare inputs & print terminal commands"] {type:"string"}

#@markdown ### Model & Paths
#@markdown Leave as `Auto (from Setup)` to use the model installed in the Setup cell.
model_choice = "Auto (from Setup)" #@param ["Auto (from Setup)", "AlphaFold3", "OpenFold3", "Boltz2", "RosettaFold3"] {type:"string"}
mpnn_dir = "unconditional_symmetry_test_example/mpnn_out" #@param {type:"string"}
#@markdown Leave blank to auto-detect from `SYMMETRY_STATE` or `<mpnn_dir>/../rfd3_out` (used to read exact chain counts/lengths).
rfd3_dir = "" #@param {type:"string"}
#@markdown Leave blank to auto-name (`unconditional_symmetry_test_example/<model>_out`)
out_dir = "" #@param {type:"string"}
#@markdown Which designed sequence to refold per backbone (`1` = first designed sequence; skips entry 0 if native/poly-Gly).
seq_index = 1 #@param {type:"integer"}
#@markdown Number of subunits (`0` = auto-detect from RFD3 backbone, `SYMMETRY_STATE`, or filename like `_C3_`/`_D2_`).
n_subunits = 0 #@param {type:"integer"}
on_existing = "overwrite" #@param ["overwrite", "skip"] {type:"string"}

#@markdown ### Shared Sampling Settings
#@markdown `num_recycles`: `10` is standard for AF3/OpenFold3/RF3; `3` is the Boltz-2 default (faster) but `10` is recommended to match AF3.
num_recycles = 10 #@param {type:"integer"}
num_diffusion_samples = 5 #@param {type:"integer"}
#@markdown Comma-separated seeds (AF3/OpenFold3 use all seeds; Boltz-2 and RF3 use the first seed).
seeds = "1" #@param {type:"string"}

#@markdown ### Boltz-2 Specific Options *(ignored for AF3 / OpenFold3 / RF3)*
boltz_sampling_steps = 200 #@param {type:"integer"}
boltz_step_scale = 1.5 #@param {type:"number"}
boltz_use_potentials = False #@param {type:"boolean"}
boltz_output_format = "mmcif" #@param ["mmcif", "pdb"] {type:"string"}

#@markdown ### RosettaFold3 Specific Options *(ignored for AF3 / OpenFold3 / Boltz-2)*
#@markdown Diffusion steps (`200` is standard; `50` is >2x faster with similar accuracy).
rf3_num_steps = 200 #@param {type:"integer"}
#@markdown Set to `0.0` to always output structures for single-sequence refolding (avoids RF3 early stopping).
rf3_early_stopping_plddt = 0.0 #@param {type:"number"}

import gzip, json, os, pathlib, re, shlex, shutil, subprocess, time
import biotite.structure as struc
import biotite.structure.io.pdbx as pdbx_io

# 1. Resolve selected model and validate its environment
selected = os.environ.get("FOLDING_MODEL", "AlphaFold3") if model_choice == "Auto (from Setup)" else model_choice
model_key = {
    "AlphaFold3": "alphafold3",
    "OpenFold3": "openfold3",
    "Boltz2": "boltz2",
    "RosettaFold3": "rosettafold3",
}[selected]
env_name = {"Boltz2": "boltz", "RosettaFold3": "foundry"}.get(selected, "af3")

if not (pathlib.Path(os.environ["ENVS_DIR"]) / env_name).exists():
  raise RuntimeError(f"Environment '{env_name}' is not installed. Select '{selected}' in the Setup cell and re-run it.")

if selected == "RosettaFold3":
  ckpt_dir = pathlib.Path(os.environ.get("FOUNDRY_CHECKPOINT_DIRS", "/content/foundry/checkpoints"))
  rf3_ckpts = list(ckpt_dir.glob("rf3*.pt")) + list(ckpt_dir.glob("rf3*.ckpt"))
  if not rf3_ckpts:
    raise RuntimeError(f"No RF3 checkpoint found in {ckpt_dir}. Select 'RosettaFold3' in the Setup cell and re-run it.")

mpnn_path = pathlib.Path(mpnn_dir).resolve()
pred_out = pathlib.Path(out_dir.strip()).resolve() if out_dir.strip() else (mpnn_path.parent / f"{model_key}_out").resolve()
inputs_dir = pred_out / "inputs"

if on_existing == "overwrite" and pred_out.exists():
  shutil.rmtree(pred_out)
inputs_dir.mkdir(parents=True, exist_ok=True)

# Resolve RFD3 backbone directory (optional, used to read exact per-chain lengths)
resolved_rfd3 = (
    pathlib.Path(rfd3_dir.strip()).resolve()
    if rfd3_dir.strip()
    else (
        pathlib.Path(SYMMETRY_STATE["rfd3_out"]).resolve()
        if ("SYMMETRY_STATE" in globals() and SYMMETRY_STATE.get("rfd3_out"))
        else (mpnn_path.parent / "rfd3_out").resolve()
    )
)

# 2. Collect designed sequences from MPNN output (supports both Foundry mpnn and standalone LigandMPNN)
fasta_files = sorted((mpnn_path / "seqs").glob("*.fa*")) if (mpnn_path / "seqs").is_dir() else []
if not fasta_files:
  fasta_files = sorted(mpnn_path.rglob("*.fa*"))
if not fasta_files:
  raise FileNotFoundError(f"No .fa/.fasta files found in {mpnn_path}")

def _parse_fasta(path: pathlib.Path) -> list[tuple[str, str]]:
  entries, header, chunks = [], None, []
  for raw in path.read_text().splitlines():
    line = raw.strip()
    if not line:
      continue
    if line.startswith(">"):
      if header is not None:
        entries.append((header, "".join(chunks)))
      header, chunks = line[1:], []
    else:
      chunks.append(line)
  if header is not None:
    entries.append((header, "".join(chunks)))
  return entries

def _backbone_chain_lengths(stem: str) -> list[int]:
  """Read per-chain residue counts from the matching RFD3 .cif / .cif.gz backbone if available."""
  if not resolved_rfd3.is_dir():
    return []
  clean = stem
  for ext in (".gz", ".cif", ".pdb"):
    clean = clean.removesuffix(ext)
  cif = next(
      (p for p in (resolved_rfd3 / f"{clean}.cif.gz", resolved_rfd3 / f"{clean}.cif") if p.is_file()),
      None,
  )
  if cif is None:
    return []
  opener = gzip.open if cif.suffix == ".gz" else open
  with opener(cif, "rt") as f:
    atoms = pdbx_io.get_structure(pdbx_io.CIFFile.read(f), model=1)
  ca = atoms[(atoms.atom_name == "CA") & struc.filter_amino_acids(atoms)]
  return [int((ca.chain_id.astype(str) == cid).sum()) for cid in dict.fromkeys(ca.chain_id.astype(str))]

def _infer_n_subunits(stem: str, bb_lengths: list[int]) -> int:
  if int(n_subunits) > 0:
    return int(n_subunits)
  if bb_lengths:
    return len(bb_lengths)
  if "SYMMETRY_STATE" in globals() and SYMMETRY_STATE.get("n_subunits"):
    return int(SYMMETRY_STATE["n_subunits"])
  m = re.search(r"(?:^|_)([CD])(\d+)(?:_|$)", stem.upper())
  if m:
    order = int(m.group(2))
    return order if m.group(1) == "C" else 2 * order
  return 1

def _split_oligomer_seq(raw_seq: str, stem: str, bb_lengths: list[int]) -> list[str]:
  """Split a FASTA sequence into per-subunit chains (handles delimited, concatenated tied, concatenated untied, or single-monomer)."""
  cleaned = re.sub(r"\s+", "", raw_seq).upper()
  n_sub = _infer_n_subunits(stem, bb_lengths)

  if ":" in cleaned or "/" in cleaned:
    parts = [p for p in re.split(r"[:/]", cleaned) if p]
    if len(parts) == 1 and n_sub > 1:
      return parts * n_sub
    return parts

  if bb_lengths:
    if len(cleaned) == sum(bb_lengths):
      out, pos = [], 0
      for L in bb_lengths:
        out.append(cleaned[pos : pos + L])
        pos += L
      return out
    if len(cleaned) == bb_lengths[0] and len(set(bb_lengths)) == 1:
      return [cleaned] * len(bb_lengths)

  if n_sub > 1 and len(cleaned) % n_sub == 0:
    L = len(cleaned) // n_sub
    return [cleaned[i * L : (i + 1) * L] for i in range(n_sub)]

  return [cleaned] * max(1, n_sub)

CHAIN_IDS = list("ABCDEFGHIJKLMNOPQRSTUVWXYZ")
designs = []  # list of (stem, [chain_seq_A, chain_seq_B, ...])
tied_count, untied_count = 0, 0
for fa in fasta_files:
  entries = _parse_fasta(fa)
  if not entries:
    continue
  has_native_entry0 = "b0_d0" not in entries[0][0]
  idx = min(seq_index if has_native_entry0 else max(0, seq_index - 1), len(entries) - 1)
  bb_lengths = _backbone_chain_lengths(fa.stem)
  chain_seqs = _split_oligomer_seq(entries[idx][1], fa.stem, bb_lengths)
  if len(chain_seqs) > len(CHAIN_IDS):
    raise ValueError(f"{fa.stem} has {len(chain_seqs)} chains (max supported: {len(CHAIN_IDS)}).")
  if len(set(chain_seqs)) == 1:
    tied_count += 1
  else:
    untied_count += 1
  designs.append((fa.stem, chain_seqs))

seed_list = sorted({int(s) for s in re.findall(r"\d+", seeds)}) or [1]
n_chains_sample = len(designs[0][1]) if designs else 0
sub_len_sample = len(designs[0][1][0]) if designs and designs[0][1] else 0
print(
    f"Prepared {len(designs)} oligomer(s) (e.g., {n_chains_sample} chains × {sub_len_sample} aa; "
    f"{tied_count} tied / {untied_count} untied) for {selected} -> {inputs_dir}",
    flush=True,
)

def _emit_terminal_script(env_label: str, exports: dict[str, str], cmd_args: list[str], script_name: str):
  env_path = pathlib.Path(os.environ["ENVS_DIR"]) / env_label
  lines = [
      "#!/usr/bin/env bash",
      "set -euo pipefail",
      "",
      f"# Activate '{env_label}' environment & set required paths",
      f"export VIRTUAL_ENV={shlex.quote(str(env_path))}",
      f'export PATH="{env_path}/bin:$PATH"',
  ]
  for k, v in exports.items():
    lines.append(f"export {k}={shlex.quote(str(v))}")
  lines.append("")
  lines.append(f"# Run {selected}")
  lines.append(" \\\n  ".join(shlex.quote(str(c)) for c in cmd_args))
  script = "\n".join(lines) + "\n"

  sh_path = pred_out.parent / script_name
  sh_path.write_text(script)
  sh_path.chmod(0o755)
  print("\n" + "=" * 72)
  print("TERMINAL INSTRUCTIONS — Copy & paste into a terminal, or run:")
  print(f"  bash {sh_path}")
  print("=" * 72)
  print(script)

# 3. Prepare model-specific inputs, print config preview, and either print CLI instructions or run
if selected in ("AlphaFold3", "OpenFold3"):
  written_jsons = []
  for stem, chain_seqs in designs:
    job_name = stem.lower()
    entities = [
        {
            "protein": {
                "id": CHAIN_IDS[c_idx],
                "sequence": seq,
                "unpairedMsa": f">query\n{seq}\n",
                "pairedMsa": "",
                "templates": [],
            }
        }
        for c_idx, seq in enumerate(chain_seqs)
    ]
    fold_input = {
        "name": job_name,
        "sequences": entities,
        "modelSeeds": seed_list,
        "dialect": "alphafold3",
        "version": 1,
    }
    json_path = inputs_dir / f"{job_name}.json"
    json_path.write_text(json.dumps(fold_input, indent=2) + "\n")
    written_jsons.append(json_path)

  if written_jsons:
    extra = f" (showing 1 of {len(written_jsons)})" if len(written_jsons) > 1 else ""
    print(f"\nInput JSON saved to {written_jsons[0]}{extra}:")
    print(written_jsons[0].read_text(), flush=True)

  cap_out = subprocess.run(
      ["nvidia-smi", "--query-gpu=compute_cap", "--format=csv,noheader"],
      capture_output=True, text=True,
  ).stdout.split()
  cap = float(cap_out[0]) if cap_out else 0.0

  if cap < 8.0:                   # T4 / V100
    flash_impl, xla_flag = "xla", "--xla_disable_hlo_passes=custom-kernel-fusion-rewriter"
  elif 8.0 < cap < 9.0:           # L4 / Ada (cc 8.6 / 8.9)
    flash_impl, xla_flag = "xla", "--xla_gpu_enable_triton_gemm=false"
  else:                           # A100 (8.0) / H100 (9.0+)
    flash_impl, xla_flag = "triton", "--xla_gpu_enable_triton_gemm=false"

  xla_flags = os.environ.get("XLA_FLAGS", "")
  if xla_flag not in xla_flags:
    xla_flags = f"{xla_flags} {xla_flag}".strip()
  os.environ["XLA_FLAGS"] = xla_flags

  af3_bin = pathlib.Path(os.environ["ENVS_DIR"]) / "af3" / "bin"
  cmd = [
      str(af3_bin / "python"), f"{os.environ['AF3_DIR']}/run_alphafold.py",
      "--stderrthreshold=error",
      f"--input_dir={inputs_dir}",
      f"--model={model_key}",
      "--norun_data_pipeline",
      f"--output_dir={pred_out}",
      f"--cache_dir={os.environ['AF3_CACHE_DIR']}",
      "--force_output_dir",
      f"--flash_attention_implementation={flash_impl}",
      f"--num_recycles={max(1, int(num_recycles))}",
      f"--num_diffusion_samples={max(1, int(num_diffusion_samples))}",
  ]
  if selected == "AlphaFold3":
    cmd.append(f"--model_dir={os.environ['AF3_WEIGHTS_DIR']}")

  if execution_mode == "Prepare inputs & print terminal commands":
    _emit_terminal_script(
        "af3",
        {
            "XLA_FLAGS": xla_flags,
            "TF_CPP_MIN_LOG_LEVEL": "3",
            "ABSL_MIN_LOG_LEVEL": "2",
            "GLOG_minloglevel": "2",
        },
        ["python", *cmd[1:]],
        f"run_{model_key}.sh",
    )
  else:
    print(f"[af3] {' '.join(cmd)}", flush=True)
    env = os.environ.copy()
    env["PATH"] = f"{af3_bin}:{env.get('PATH', '')}"
    env["TF_CPP_MIN_LOG_LEVEL"] = "3"
    env["ABSL_MIN_LOG_LEVEL"] = "2"
    env["GLOG_minloglevel"] = "2"
    env["PYTHONUNBUFFERED"] = "1"

    noise = ("cuda_timer.cc:", "TOKAMAX autotune unavailable")
    with subprocess.Popen(
        cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, env=env
    ) as proc:
      skip_tb = False
      for line in proc.stdout:
        if any(k in line for k in noise):
          skip_tb = "TOKAMAX" in line
          continue
        if skip_tb:
          if line.startswith(("  File ", "    ", "Traceback ", "AttributeError:", "RuntimeError:")):
            continue
          skip_tb = False
        print(line, end="", flush=True)

      if proc.wait() != 0:
        raise subprocess.CalledProcessError(proc.returncode, cmd)

elif selected == "Boltz2":
  written_yamls = []
  for stem, chain_seqs in designs:
    lines = ["version: 1", "sequences:"]
    for c_idx, seq in enumerate(chain_seqs):
      lines.append("- protein:")
      lines.append(f"    id: {CHAIN_IDS[c_idx]}")
      lines.append(f"    sequence: {seq}")
      lines.append("    msa: empty")
    yaml_path = inputs_dir / f"{stem}.yaml"
    yaml_path.write_text("\n".join(lines) + "\n")
    written_yamls.append(yaml_path)

  if written_yamls:
    extra = f" (showing 1 of {len(written_yamls)})" if len(written_yamls) > 1 else ""
    print(f"\nInput YAML saved to {written_yamls[0]}{extra}:")
    print(written_yamls[0].read_text(), flush=True)

  cmd = [
      "boltz", "predict", str(inputs_dir),
      "--cache", os.environ["BOLTZ_CACHE"],
      "--out_dir", str(pred_out),
      "--recycling_steps", str(max(1, int(num_recycles))),
      "--sampling_steps", str(max(1, int(boltz_sampling_steps))),
      "--diffusion_samples", str(max(1, int(num_diffusion_samples))),
      "--step_scale", str(float(boltz_step_scale)),
      "--output_format", boltz_output_format,
  ]
  if boltz_use_potentials:
    cmd.append("--use_potentials")
  if on_existing == "overwrite":
    cmd.append("--override")

  if execution_mode == "Prepare inputs & print terminal commands":
    _emit_terminal_script(
        "boltz",
        {"BOLTZ_CACHE": os.environ["BOLTZ_CACHE"]},
        cmd,
        "run_boltz2.sh",
    )
  else:
    print(f"[boltz] {' '.join(cmd)}", flush=True)
    boltz_bin = pathlib.Path(os.environ["ENVS_DIR"]) / "boltz" / "bin"
    cmd[0] = str(boltz_bin / "boltz")

    env = os.environ.copy()
    env["PATH"] = f"{boltz_bin}:{env.get('PATH', '')}"

    log_path = pred_out / "boltz.log"
    ext = "pdb" if boltz_output_format == "pdb" else "cif"
    seen, total, t0 = set(), len(designs), time.time()

    print(f"Running Boltz-2 on {total} oligomer(s) (log: {log_path.name})...", flush=True)
    with open(log_path, "w") as log_f:
      proc = subprocess.Popen(cmd, stdout=log_f, stderr=subprocess.STDOUT, env=env)
      while proc.poll() is None:
        for p in sorted(pred_out.rglob(f"*_model_0.{ext}")):
          if not p.is_file():
            continue
          name = p.parent.name
          if name not in seen:
            seen.add(name)
            print(f"  [{len(seen)}/{total}] {name} ({time.time() - t0:.0f}s elapsed)", flush=True)
        time.sleep(2)

    for p in sorted(pred_out.rglob(f"*_model_0.{ext}")):
      if not p.is_file():
        continue
      name = p.parent.name
      if name not in seen:
        seen.add(name)
        print(f"  [{len(seen)}/{total}] {name} ({time.time() - t0:.0f}s elapsed)", flush=True)

    if proc.returncode != 0:
      print(log_path.read_text()[-3000:])
      raise subprocess.CalledProcessError(proc.returncode, cmd)

elif selected == "RosettaFold3":
  rf3_entries = [
      {
          "name": stem,
          "components": [
              {"seq": seq, "chain_id": CHAIN_IDS[c_idx]}
              for c_idx, seq in enumerate(chain_seqs)
          ],
      }
      for stem, chain_seqs in designs
  ]
  rf3_json = inputs_dir / "rf3_inputs.json"
  rf3_json.write_text(json.dumps(rf3_entries, indent=2) + "\n")

  print(f"\nInput JSON saved to {rf3_json} ({len(rf3_entries)} design(s)):")
  print(rf3_json.read_text(), flush=True)

  cmd = [
      "rf3", "fold",
      f"inputs={rf3_json}",
      f"out_dir={pred_out}",
      f"n_recycles={max(1, int(num_recycles))}",
      f"diffusion_batch_size={max(1, int(num_diffusion_samples))}",
      f"num_steps={max(1, int(rf3_num_steps))}",
      f"early_stopping_plddt_threshold={float(rf3_early_stopping_plddt)}",
      f"seed={seed_list[0]}",
  ]
  if on_existing == "skip":
    cmd.append("skip_existing=True")

  if execution_mode == "Prepare inputs & print terminal commands":
    _emit_terminal_script(
        "foundry",
        {
            "FOUNDRY_CHECKPOINT_DIRS": os.environ["FOUNDRY_CHECKPOINT_DIRS"],
            "CCD_MIRROR_PATH": "/tmp",
            "PDB_MIRROR_PATH": "/tmp",
            "PYTHONWARNINGS": "ignore",
        },
        cmd,
        "run_rosettafold3.sh",
    )
  else:
    run_command_with_env("foundry", cmd)

if execution_mode == "Run in Colab":
  print(f"\nDone! Predictions saved to: {pred_out}", flush=True)

In [ ]:
#@title Visualize Symmetric RFD3 Designs vs. Predicted Oligomers
#@markdown Interactively compare each RFD3 oligomer backbone (grey) against its refolded prediction(s) with permutation-invariant complex & monomer Cα RMSD.

design_glob = "unconditional_symmetry_test_example/rfd3_out/*.cif.gz" #@param {type:"string"}
#@markdown Leave `pred_dir` blank to automatically use `pred_out` from the refolding cell above, or enter a custom path.
pred_dir = "" #@param {type:"string"}
#@markdown Show only the top-ranked model per design (`True`) or superimpose up to `max_models` samples (`False`).
only_best = True #@param {type:"boolean"}
#@markdown Color each predicted subunit distinctly when `only_best=True` (ignored when overlaying multiple samples).
color_by_chain = True #@param {type:"boolean"}
max_models = 5 #@param {type:"integer"}
#@markdown Set the size of the visualization
width = 700 #@param {type:"integer"}
height = 500 #@param {type:"integer"}

target_pred = pred_dir.strip() or (str(pred_out) if "pred_out" in globals() else "")
if not target_pred:
  raise ValueError("No prediction folder specified. Run the refolding cell first or enter a path in pred_dir.")

browse_designs(
    design_glob=design_glob,
    pred_root=target_pred,
    only_best=only_best,
    color_by_chain=color_by_chain,
    max_models=max_models,
    width=width,
    height=height,
)

In [ ]:
#@title Evaluate & Plot Symmetric Oligomer Designability (AF3 / OpenFold3 / Boltz-2 / RF3)
#@markdown Leave `pred_root` blank to automatically use `pred_out` from the refolding cell above, or enter a custom path.
design_glob = "unconditional_symmetry_test_example/rfd3_out/*.cif.gz" #@param {type:"string"}
pred_root = "" #@param {type:"string"}
only_best = True #@param {type:"boolean"}
out_prefix = "unconditional_symmetry_test_example/symmetry_eval" #@param {type:"string"}

#@markdown ### Designability Cutoffs (for dashed guide lines & pass/fail coloring)
complex_rmsd_cutoff = 2.0 #@param {type:"number"}
iptm_cutoff = 0.75 #@param {type:"number"}
plddt_cutoff = 80.0 #@param {type:"number"}
inter_pae_cutoff = 5.0 #@param {type:"number"}

import glob, gzip, itertools, json, pathlib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import biotite.structure as struc
import biotite.structure.io.pdb as pdb_io
import biotite.structure.io.pdbx as pdbx_io


def _strip_struct_ext(path: pathlib.Path) -> str:
  name = path.name
  for ext in (".gz", ".cif", ".pdb", ".bcif"):
    name = name.removesuffix(ext)
  return name


def _load_struct(path):
  p = str(path)
  opener = gzip.open if p.endswith(".gz") else open
  base = p[:-3] if p.endswith(".gz") else p
  if base.endswith(".cif"):
    with opener(p, "rt") as f:
      return pdbx_io.get_structure(pdbx_io.CIFFile.read(f), model=1, extra_fields=["b_factor"])
  with opener(p, "rt") as f:
    return pdb_io.PDBFile.read(f).get_structure(model=1, extra_fields=["b_factor"])


def _kabsch(P, Q):
  p_mean, q_mean = P.mean(axis=0), Q.mean(axis=0)
  H = (Q - q_mean).T @ (P - p_mean)
  U, _, Vt = np.linalg.svd(H)
  d = np.sign(np.linalg.det(Vt.T @ U.T))
  R = Vt.T @ np.diag([1.0, 1.0, d]) @ U.T
  t = p_mean - q_mean @ R.T
  Q_fit = Q @ R.T + t
  return R, t, float(np.sqrt(np.mean(np.sum((P - Q_fit) ** 2, axis=-1))))


def _chain_ca_coords(struct):
  ca = struct[(struct.atom_name == "CA") & struc.filter_amino_acids(struct)]
  cids = list(dict.fromkeys(ca.chain_id.astype(str)))
  return [ca.coord[ca.chain_id.astype(str) == cid] for cid in cids]


def _calc_oligomer_rmsds(fixed, mobile):
  """Permutation-invariant complex Cα RMSD and mean monomer Cα RMSD."""
  f_chains = _chain_ca_coords(fixed)
  m_chains = _chain_ca_coords(mobile)
  n_chains = len(f_chains)

  if n_chains >= 2 and len(m_chains) == n_chains and len({len(c) for c in f_chains + m_chains}) == 1:
    L = len(f_chains[0])
    P_all = np.vstack(f_chains)

    if n_chains <= 4:
      perms = list(itertools.permutations(range(n_chains)))
    else:
      f_cents = np.array([c.mean(axis=0) for c in f_chains])
      m_cents = np.array([c.mean(axis=0) for c in m_chains])
      perms = []
      for k in range(n_chains):
        R0, t0, _ = _kabsch(f_chains[k], m_chains[0])
        mc_aligned = m_cents @ R0.T + t0
        used, perm = set(), [None] * n_chains
        for fi in range(n_chains):
          for mi in np.argsort(np.linalg.norm(mc_aligned - f_cents[fi], axis=-1)):
            if int(mi) not in used:
              perm[fi] = int(mi)
              used.add(int(mi))
              break
        perms.append(tuple(perm))

    best_rmsd, best_perm = np.inf, None
    for perm in perms:
      Q_all = np.vstack([m_chains[i] for i in perm])
      _, _, rmsd = _kabsch(P_all, Q_all)
      if rmsd < best_rmsd:
        best_rmsd, best_perm = rmsd, perm

    mono_rmsd = float(np.mean([_kabsch(f_chains[fi], m_chains[mi])[2] for fi, mi in enumerate(best_perm)]))
    return {
        "n_chains": n_chains,
        "subunit_len": L,
        "complex_rmsd": float(best_rmsd),
        "monomer_rmsd": mono_rmsd,
    }

  # Fallback for single chain or mismatched lengths
  f_ca = fixed[(fixed.atom_name == "CA") & struc.filter_amino_acids(fixed)]
  m_ca = mobile[(mobile.atom_name == "CA") & struc.filter_amino_acids(mobile)]
  if len(f_ca) == len(m_ca) and len(f_ca) >= 3:
    _, _, rmsd = _kabsch(f_ca.coord, m_ca.coord)
    return {"n_chains": n_chains, "subunit_len": len(f_ca), "complex_rmsd": rmsd, "monomer_rmsd": rmsd}
  res = struc.superimpose_structural_homologs(f_ca, m_ca)
  tf, f_idx, m_idx = res[1:] if len(res) == 4 else res
  rmsd = float(struc.rmsd(f_ca[f_idx], tf.apply(m_ca[m_idx])))
  return {"n_chains": n_chains, "subunit_len": len(f_idx), "complex_rmsd": rmsd, "monomer_rmsd": rmsd}


def _mean_plddt(mobile):
  ca = mobile[(mobile.atom_name == "CA") & struc.filter_amino_acids(mobile)]
  if len(ca) == 0 or not hasattr(ca, "b_factor"):
    return np.nan
  vals = np.asarray(ca.b_factor, dtype=float)
  return float(np.mean(vals) * (100.0 if np.nanmax(vals) <= 1.0 else 1.0))


def _parse_oligomer_confidences(model_path: pathlib.Path, mobile):
  parent, stem = model_path.parent, _strip_struct_ext(model_path)
  base_stem = stem.removesuffix("_model")
  res = {"iptm": np.nan, "ptm": np.nan, "inter_pae": np.nan, "min_inter_pae": np.nan, "has_clash": 0}

  # 1. Summary JSON (AF3/OF3/RF3 summary_confidences.json or Boltz-2 confidence_*.json)
  summary_file = next((
      p for p in (
          parent / f"{base_stem}_summary_confidences.json",
          parent / "summary_confidences.json",
          parent / f"confidence_{stem}.json",
      ) if p.exists()
  ), None)
  if summary_file is not None:
    data = json.loads(summary_file.read_text())
    if data.get("iptm") is not None:
      res["iptm"] = float(data["iptm"])
    if data.get("ptm") is not None:
      res["ptm"] = float(data["ptm"])
    res["has_clash"] = int(data.get("has_clash", 0))
    cp_pae = data.get("chain_pair_pae_min")
    if isinstance(cp_pae, list) and len(cp_pae) >= 2:
      off_diag = [
          float(cp_pae[i][j])
          for i in range(len(cp_pae))
          for j in range(len(cp_pae[i]))
          if i != j and cp_pae[i][j] is not None
      ]
      if off_diag:
        res["min_inter_pae"] = float(np.min(off_diag))

  # 2. Full PAE matrix (AF3/OF3/RF3 confidences.json or Boltz-2 pae_*.npz)
  pae, cids = None, None
  conf_json = next((
      p for p in (
          parent / f"{base_stem}_confidences.json",
          parent / "confidences.json",
      ) if p.exists()
  ), None)
  if conf_json is not None:
    data = json.loads(conf_json.read_text())
    if "pae" in data:
      pae = np.asarray(data["pae"], dtype=float)
      if "token_chain_ids" in data:
        cids = np.asarray(data["token_chain_ids"], dtype=str)
  else:
    boltz_pae = parent / f"pae_{stem}.npz"
    if boltz_pae.exists():
      pae = np.asarray(np.load(boltz_pae)["pae"], dtype=float)

  if pae is not None:
    if cids is None or len(cids) != pae.shape[0]:
      ca = mobile[(mobile.atom_name == "CA") & struc.filter_amino_acids(mobile)]
      if len(ca) == pae.shape[0]:
        cids = ca.chain_id.astype(str)
    if cids is not None and len(cids) == pae.shape[0]:
      inter_mask = cids[:, None] != cids[None, :]
      if np.any(inter_mask):
        res["inter_pae"] = float(np.mean(pae[inter_mask]))
        if np.isnan(res["min_inter_pae"]):
          res["min_inter_pae"] = float(np.min(pae[inter_mask]))
    else:
      res["inter_pae"] = float(np.mean(pae))

  return res


# Resolve pred_root (auto-fallback to pred_out from the refolding cell)
resolved_pred = pred_root.strip() or (str(pred_out) if "pred_out" in globals() else "")
if not resolved_pred:
  raise ValueError("No prediction directory specified. Run the refolding cell first or set pred_root.")
pred_path = pathlib.Path(resolved_pred).resolve()

# Collect metrics across all designs
rows = []
for design_path in sorted(glob.glob(str(design_glob))):
  stem = _strip_struct_ext(pathlib.Path(design_path))

  candidates = (
      pred_path / stem, pred_path / stem.lower(),
      pred_path / f"{stem}.cif", pred_path / f"{stem}.cif".lower(),
      pred_path / "predictions" / stem, pred_path / "predictions" / f"{stem}.cif",
  )
  design_dir = next((d for d in candidates if d.is_dir()), None)
  if design_dir is not None:
    matches = [
        p for pat in ("*.cif", "*.cif.gz", "*.pdb")
        for p in sorted(design_dir.rglob(pat))
        if p.is_file() and "inputs" not in p.parts
    ]
  else:
    matches = [
        p for pat in (f"{stem}*model*.cif", f"{stem}*model*.cif.gz", f"{stem}*model*.pdb")
        for p in sorted(pred_path.rglob(pat))
        if p.is_file() and "inputs" not in p.parts
    ]

  if only_best:
    top_dir = design_dir if design_dir is not None else pred_path
    top_af3_rf3 = [m for m in matches if m.parent == top_dir and m.name.endswith(("_model.cif", "_model.cif.gz"))]
    top_boltz = [m for m in matches if m.name.endswith(("_model_0.cif", "_model_0.pdb"))]
    matches = (top_af3_rf3 or top_boltz or matches)[:1]
  else:
    samples = [m for m in matches if "sample-" in str(m) or "_model_" in m.name]
    matches = samples or matches

  if not matches:
    continue

  fixed = _load_struct(design_path)
  for m in matches:
    mobile = _load_struct(m)
    row = {
        "design": stem,
        "model": m.parent.name if m.name == "model.cif" else _strip_struct_ext(m),
    }
    row.update(_calc_oligomer_rmsds(fixed, mobile))
    row["plddt"] = _mean_plddt(mobile)
    row.update(_parse_oligomer_confidences(m, mobile))
    rows.append(row)

df = pd.DataFrame(rows)
if df.empty:
  raise FileNotFoundError(f"No matching design/prediction pairs found in {pred_path}. Check design_glob and pred_root.")

df["is_designable"] = (
    (df["complex_rmsd"] <= complex_rmsd_cutoff) &
    (df["iptm"].fillna(0.0) >= iptm_cutoff) &
    (df["plddt"] >= plddt_cutoff)
)
df = df.sort_values(by=["is_designable", "complex_rmsd", "iptm"], ascending=[False, True, False]).reset_index(drop=True)

csv_path = pathlib.Path(f"{out_prefix}_metrics.csv").resolve()
csv_path.parent.mkdir(parents=True, exist_ok=True)
df.to_csv(csv_path, index=False)
print(f"Saved {len(df)} prediction(s) to {csv_path}  |  Designable: {int(df['is_designable'].sum())}/{len(df)}")
display(df[[
    "design", "n_chains", "subunit_len", "complex_rmsd", "monomer_rmsd",
    "plddt", "iptm", "ptm", "inter_pae", "min_inter_pae", "is_designable",
]].round(2))

# --- 6-Panel Publication-Quality Plot ---
plt.rcParams.update({
    "font.family": "sans-serif",
    "font.sans-serif": ["Arial", "DejaVu Sans"],
    "axes.labelsize": 10,
    "xtick.labelsize": 9,
    "ytick.labelsize": 9,
    "axes.spines.top": False,
    "axes.spines.right": False,
})

fig, axes = plt.subplots(2, 3, figsize=(12.2, 7.2), dpi=300)
colors = np.where(df["is_designable"], "#54A24B", "#4C78A8")
labels_legend = [
    plt.Line2D([0], [0], marker="o", color="w", markerfacecolor="#54A24B", markersize=7, label="Designable (passes cutoffs)"),
    plt.Line2D([0], [0], marker="o", color="w", markerfacecolor="#4C78A8", markersize=7, label="Other design"),
]

# Panel A: Complex scRMSD vs. ipTM
ax = axes[0, 0]
ax.scatter(df["complex_rmsd"], df["iptm"], c=colors, s=55, edgecolors="white", linewidths=0.6, alpha=0.9)
ax.axvline(complex_rmsd_cutoff, color="#888888", linestyle="--", linewidth=0.9)
ax.axhline(iptm_cutoff, color="#888888", linestyle="--", linewidth=0.9)
ax.set_xlabel("Complex scRMSD (Å)")
ax.set_ylabel("ipTM")
ax.set_title("A. Assembly vs. ipTM", fontsize=10, fontweight="bold", loc="left")

# Panel B: Complex scRMSD vs. Mean pLDDT
ax = axes[0, 1]
ax.scatter(df["complex_rmsd"], df["plddt"], c=colors, s=55, edgecolors="white", linewidths=0.6, alpha=0.9)
ax.axvline(complex_rmsd_cutoff, color="#888888", linestyle="--", linewidth=0.9)
ax.axhline(plddt_cutoff, color="#888888", linestyle="--", linewidth=0.9)
ax.set_xlabel("Complex scRMSD (Å)")
ax.set_ylabel("Mean pLDDT")
ax.set_title("B. Assembly vs. pLDDT", fontsize=10, fontweight="bold", loc="left")

# Panel C: Monomer scRMSD vs. Complex scRMSD
ax = axes[0, 2]
ax.scatter(df["monomer_rmsd"], df["complex_rmsd"], c=colors, s=55, edgecolors="white", linewidths=0.6, alpha=0.9)
ax.axvline(1.5, color="#888888", linestyle="--", linewidth=0.9)
ax.axhline(complex_rmsd_cutoff, color="#888888", linestyle="--", linewidth=0.9)
lim_max = max(df["monomer_rmsd"].max(), df["complex_rmsd"].max(), 2.5) * 1.05
ax.plot([0, lim_max], [0, lim_max], color="#CCCCCC", linestyle=":", linewidth=1.0, zorder=0)
ax.set_xlabel("Monomer scRMSD (Å)")
ax.set_ylabel("Complex scRMSD (Å)")
ax.set_title("C. Subunit Fold vs. Assembly", fontsize=10, fontweight="bold", loc="left")

# Panel D: Complex scRMSD vs. Inter-chain Mean PAE
ax = axes[1, 0]
if df["inter_pae"].notna().any():
  ax.scatter(df["complex_rmsd"], df["inter_pae"], c=colors, s=55, edgecolors="white", linewidths=0.6, alpha=0.9)
  ax.axvline(complex_rmsd_cutoff, color="#888888", linestyle="--", linewidth=0.9)
  ax.axhline(inter_pae_cutoff, color="#888888", linestyle="--", linewidth=0.9)
else:
  ax.text(0.5, 0.5, "Full PAE matrix not found", ha="center", va="center", fontsize=9, color="#666666", transform=ax.transAxes)
ax.set_xlabel("Complex scRMSD (Å)")
ax.set_ylabel("Inter-chain Mean PAE (Å)")
ax.set_title("D. Assembly vs. Inter-chain PAE", fontsize=10, fontweight="bold", loc="left")

# Panel E: Mean pLDDT vs. ipTM
ax = axes[1, 1]
ax.scatter(df["plddt"], df["iptm"], c=colors, s=55, edgecolors="white", linewidths=0.6, alpha=0.9)
ax.axvline(plddt_cutoff, color="#888888", linestyle="--", linewidth=0.9)
ax.axhline(iptm_cutoff, color="#888888", linestyle="--", linewidth=0.9)
ax.set_xlabel("Mean pLDDT")
ax.set_ylabel("ipTM")
ax.set_title("E. Local vs. Interface Confidence", fontsize=10, fontweight="bold", loc="left")

# Panel F: Complex scRMSD vs. pTM
ax = axes[1, 2]
ax.scatter(df["complex_rmsd"], df["ptm"], c=colors, s=55, edgecolors="white", linewidths=0.6, alpha=0.9)
ax.axvline(complex_rmsd_cutoff, color="#888888", linestyle="--", linewidth=0.9)
ax.axhline(0.80, color="#888888", linestyle="--", linewidth=0.9)
ax.set_xlabel("Complex scRMSD (Å)")
ax.set_ylabel("pTM")
ax.set_title("F. Assembly vs. Global pTM", fontsize=10, fontweight="bold", loc="left")

for ax in axes.flat:
  ax.tick_params(top=False, right=False)

fig.legend(handles=labels_legend, loc="upper center", ncol=2, frameon=False, fontsize=9, bbox_to_anchor=(0.5, 1.02))
plt.tight_layout(rect=[0, 0, 1, 0.97])

fig_path = pathlib.Path(f"{out_prefix}_summary.png").resolve()
plt.savefig(fig_path, dpi=300, bbox_inches="tight")
plt.show()
print(f"Figure saved to {fig_path}")

In [ ]:
#@title Extract & Plot RFD3 Backbone Metrics (Unconditional Symmetry)
rfd3_dir = "unconditional_symmetry_test_example/rfd3_out" #@param {type:"string"}
eval_csv = "unconditional_symmetry_test_example/symmetry_eval_metrics.csv" #@param {type:"string"}
out_prefix = "unconditional_symmetry_test_example/rfd3_metrics" #@param {type:"string"}
#@markdown Comma-separated exact metric keys to plot. Leave blank to auto-select up to 6.
plot_metrics = "loop_fraction, helix_fraction, sheet_fraction, radius_of_gyration, num_ss_elements, max_ca_deviation" #@param {type:"string"}

import json, pathlib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.ticker import MaxNLocator

rfd3_path = pathlib.Path(rfd3_dir).resolve()
json_files = sorted(rfd3_path.glob("*.json"))
if not json_files:
  raise FileNotFoundError(f"No RFD3 .json metric files found in {rfd3_path}")

def _flatten_metrics(obj, prefix=""):
  """Flatten nested dict while preserving original metric field names."""
  flat = {}
  for k, v in obj.items():
    if isinstance(v, dict):
      sub_prefix = "" if k in ("metrics", "summary", "scores") else f"{prefix}{k}."
      flat.update(_flatten_metrics(v, sub_prefix))
    elif isinstance(v, (int, float)) and not isinstance(v, bool):
      flat[f"{prefix}{k}"] = float(v)
  return flat

rfd3_rows = []
for jf in json_files:
  data = json.loads(jf.read_text())
  row = {"design": jf.stem}
  row.update(_flatten_metrics(data))
  rfd3_rows.append(row)

df_rfd3 = pd.DataFrame(rfd3_rows)
rfd3_numeric = [c for c in df_rfd3.columns if c != "design" and df_rfd3[c].notna().any()]
print(f"Found {len(df_rfd3)} RFD3 JSON file(s) with {len(rfd3_numeric)} numeric metric(s):")
print(" ", ", ".join(rfd3_numeric))

# Merge with refolding evaluation CSV if available
eval_path = pathlib.Path(eval_csv).resolve()
if eval_path.exists():
  df_eval = pd.read_csv(eval_path)
  df = df_eval.merge(df_rfd3, on="design", how="left")
  has_refold = True
else:
  df = df_rfd3.copy()
  df["is_designable"] = False
  has_refold = False
  print(f"Note: {eval_path} not found — plotting RFD3 metrics alone.")

merged_csv = pathlib.Path(f"{out_prefix}_merged.csv").resolve()
df.to_csv(merged_csv, index=False)
print(f"Saved merged table to {merged_csv}")

all_numeric = [c for c in df.columns if c not in ("design", "model", "is_designable") and pd.api.types.is_numeric_dtype(df[c]) and df[c].notna().any()]

if plot_metrics.strip():
  requested = [m.strip() for m in plot_metrics.split(",") if m.strip()]
  missing = [m for m in requested if m not in all_numeric]
  if missing:
    print(f"Warning: these plot_metrics were not found: {missing}")
  chosen = [m for m in requested if m in all_numeric]
else:
  priority_terms = ("loop_fraction", "helix_fraction", "sheet_fraction", "radius_of_gyration", "num_ss_elements", "max_ca_deviation")
  chosen = [c for c in priority_terms if c in all_numeric][:6]
  for col in rfd3_numeric:
    if len(chosen) == 6:
      break
    if col not in chosen and df[col].nunique() > 1:
      chosen.append(col)

if not chosen:
  raise ValueError("No valid numeric metrics found to plot. Check the printed key list above.")

# --- Plot ---
plt.rcParams.update({
    "font.family": "sans-serif",
    "font.sans-serif": ["Arial", "DejaVu Sans"],
    "axes.labelsize": 10,
    "xtick.labelsize": 9,
    "ytick.labelsize": 9,
    "axes.spines.top": False,
    "axes.spines.right": False,
})

n_panels = len(chosen)
ncols = 3 if n_panels >= 5 else (2 if n_panels > 1 else 1)
nrows = int(np.ceil(n_panels / ncols))
fig, axes = plt.subplots(nrows, ncols, figsize=(4.1 * ncols, 3.5 * nrows), dpi=300, squeeze=False)
colors = np.where(df.get("is_designable", False), "#54A24B", "#4C78A8")

for ax, col in zip(axes.flat, chosen):
  if has_refold and "complex_rmsd" in df.columns and df["complex_rmsd"].notna().any():
    ax.scatter(df[col], df["complex_rmsd"], c=colors, s=55, edgecolors="white", linewidths=0.6, alpha=0.9)
    ax.set_ylabel("Complex scRMSD (Å)")
  else:
    ax.hist(df[col].dropna(), bins=15, color="#4C78A8", edgecolor="white", alpha=0.9)
    ax.set_ylabel("Count")
  ax.set_xlabel(col)
  if any(k in col.lower() for k in ("clash", "chainbreak", "num_")):
    ax.xaxis.set_major_locator(MaxNLocator(integer=True))
  ax.tick_params(top=False, right=False)

for ax in axes.flat[n_panels:]:
  ax.axis("off")

plt.tight_layout()
fig_path = pathlib.Path(f"{out_prefix}_plots.png").resolve()
plt.savefig(fig_path, dpi=300, bbox_inches="tight")
plt.show()
print(f"Figure saved to {fig_path}")